# 57. 统计折线图（lineplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 14 / 20 步：探索变量关系与趋势**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 散点图（scatterplot）  →  **本章任务：** 统计折线图（lineplot）  →  **下一步：** 回归图（regplot / lmplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

当数据是按时间或某种顺序排列的，你想看的常常是"整体走势"，而不是一个个零散的点——一张表里同一天有几十笔销售，直接画散点反而看不清重心。



## 本章目标

学完本章，你将能够：

- **理解**：理解「统计折线图（lineplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「统计折线图（lineplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「统计折线图（lineplot）」并读出其中的结论。


## 57.1 适用场景

**背景引入**：当数据是按时间或某种顺序排列的，你想看的常常是"整体走势"，而不是一个个零散的点——一张表里同一天有几十笔销售，直接画散点反而看不清重心。统计折线图会自动把同一位置的多条观察聚合成一条代表中心趋势的线，顺带画上不确定性范围。营销日活动、每周的库存变化、实验里不同样本的重复测量，想一眼读懂平均值和波动，正是这种图表最顺手的地方。

打个比方：lineplot 像'给每天的客流算一条代表线'——同一天几十笔订单本来是一堆散点，它把同一 X 位置的观察聚成一个'当天的代表值'连成线，再画一片阴影表示波动。所以它画的是'重心和范围'，不是一条条原始点。

时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。


## 57.2 数据结构

长表，一列有序X、一列数值Y，可增加分组列。


## 57.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 errorbar=None 改为 errorbar="sd" 或 errorbar=("ci", 95)，观察误差区间的显示
2. 修改 estimator 为 "median"，对比均值线与中位数线的趋势差异
3. 添加 markers=False 参数，说明标记点对时间序列可读性的作用


## 57.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.lineplot()`、`ax.set()`、`ax.tick_params()` | 时间或有序X轴上，每个位置存在多条观察，需要展示均值与不确定性。 | 把聚合线当成单个真实序列 |
| 进阶变体 | `plt.subplots()`、`sns.lineplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 日期未排序 |
| 关键参数 | `estimator` | 聚合函数 | 把聚合线当成单个真实序列 |
| 关键参数 | `errorbar` | 误差 | 日期未排序 |
| 关键参数 | `units` | 个体线 | 误差阴影含义不清 |
| 关键参数 | `sort` | 排序 | 把聚合线当成单个真实序列 |


## 57.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-57 -->
### 数学推导｜均值的不确定性区间

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜样本均值存在抽样波动。** 独立同分布条件下 $\operatorname{Var}(\bar X)=\sigma^2/n$。

**第 2 步｜用样本标准差估计未知的 $\sigma$。** 得到 $SE\approx s/\sqrt n$。

**第 3 步｜用标准化分布给出区间。** 大样本近似下

$$
\frac{\bar X-\mu}{SE}\approx N(0,1)
$$

标准正态中约 95% 落在 $[-1.96,1.96]$，移项后得到 $\bar x\pm1.96SE$。小样本时应把 1.96 换成相应的 $t$ 分位数。

**把上面的关系收束为本章计算式：**

$$
CI_{95\%}\approx \bar{x}\pm1.96\frac{s}{\sqrt{n}}
$$

**符号解释：** $\bar{x}$ 是样本均值，$s$ 是样本标准差，$n$ 是样本量。

**代码对应：** 统计图中的误差线应明确表示标准差、标准误还是置信区间。

**使用边界：** 该近似依赖样本与分布条件；小样本或偏态数据可考虑 bootstrap。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 57.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(
    data=daily,
    x="date",
    y="sales",
    ci=None,
    color="#1a73e8",
    marker="o",
    ax=ax,
)
ax.set(title="每日平均销售额", xlabel="日期", ylabel="销售额")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


**练一练**：把上一节的 `daily` 换成另一份可复现数据，看看"改一个数据字段"会让折线怎么变。请在下方用自己的代码画出：以 `orders` 为数据源，在横轴 `category`（品类组合）、纵轴 `order_value`（下单金额）上画一条平均值的折线，并把画布保存为 `ax_ex`。回想基础图表用了哪些参数（`sns.lineplot`、`ax.set`、`plt.show`）。先把 `____` 填好，再运行；若 `____` 还没替换会触发 `NameError`，那是脚手架在提醒你还没写，属正常。填完运行后，把 `estimator` 从 `mean` 改成 `median` 再运行一次，比较两条线的位置差异，并把观察写进 `fill_in`。想好答案后再看"参考解答"单元格。


In [ ]:
# 请在下方填写代码（填空式脚手架）。
# ____ 处需要替换为实际代码，运行前先想清楚会看到什么。
import seaborn as sns
import matplotlib.pyplot as plt


## 57.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(
    data=daily,
    x="date",
    y="sales",
    hue="region",
    marker="o",
    ci=None,
    palette="colorblind",
    ax=ax,
)
ax.set(title="区域每日销售趋势", xlabel="日期", ylabel="销售额")
ax.legend(title="区域", frameon=False)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()


## 57.8 参数说明

- estimator：聚合函数
- errorbar：误差
- units：个体线
- sort：排序


## 57.9 结果解读

默认线是各X位置的均值，阴影是误差区间；先确认聚合口径。


## 57.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 57.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 57.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 57.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 57.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 57.12 易错点提醒

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


## 57.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 57.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：按区域分层折线，比较不同区域的时间走势
# 【目标】用颜色加一条分组维度，比较不同区域在时间上的走势差异。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="region"，用不同颜色区分区域走势。
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(
    data=daily, x="date", y="sales", hue="region", ci=None, marker="o", ax=ax
)
ax.set(title="分区域每日销售额", xlabel="日期", ylabel="销售额")
ax.legend(title="区域", frameon=False)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

# ---- 反思记录：区域间走势是否同步，谁更高/更稳 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

weekly = daily.copy()
weekly["day"] = weekly["date"].dt.day
fig, ax = plt.subplots(figsize=(8.5, 4.3))
sns.lineplot(
    data=weekly,
    x="day",
    y="sales",
    hue="region",
    style="region",
    markers=True,
    dashes=False,
    ci=None,
    ax=ax,
)
ax.set(title="按日序号比较区域趋势", xlabel="日", ylabel="销售额")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 57.15 小结

用lineplot对重复观察进行统计聚合并显示时间趋势和误差。


### 57.15.1 你已经掌握

- 判断统计折线图（lineplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 57.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 聚合函数 |
| `errorbar` | 误差 |
| `units` | 个体线 |
| `sort` | 排序 |


### 57.15.3 需要注意

- 把聚合线当成单个真实序列
- 日期未排序
- 误差阴影含义不清


### 57.15.4 完成检查

- [ ] 能判断什么问题适合使用统计折线图（lineplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 57.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
